In [ ]:
import pandas as pd import numpy as np import os os.environ["LOKY_MAX_CPU_COUNT"] = "4"
from sklearn.preprocessing import StandardScaler from imblearn.over_sampling import SMOTE from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression from sklearn.ensemble import RandomForestClassifier, StackingClassifier from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
import matplotlib.pyplot as plt import seaborn as sns import joblib
df = pd.read_csv("creditcard.csv") print("Dataset shape:", df.shape) print("Missing values:", df.isnull().sum().max()) print("\nClass distribution:\n", df['Class'].value_counts()) print("\nPercentage distribution:\n", df['Class'].value_counts(normalize=True) * 100)
scaler = StandardScaler() df[['Amount', 'Time']] = scaler.fit_transform(df[['Amount', 'Time']])
X = df.drop('Class', axis=1) y = df['Class']
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )
sm = SMOTE(random_state=42, sampling_strategy=1.0) X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print("\nAfter SMOTE resampling:") print("Training set shape:", X_train_res.shape) print("Fraud ratio after resampling:", y_train_res.mean())
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42) rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1) xgb = XGBClassifier( n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", use_label_encoder=False, random_state=42 )
stacked_model = StackingClassifier( estimators=[ ('log_reg', log_reg), ('rf', rf), ('xgb', xgb) ], final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"), n_jobs=-1 )
stacked_model.fit(X_train_res, y_train_res)
def find_best_threshold(model, X_val, y_val): probs = model.predict_proba(X_val)[:, 1] thresholds = np.linspace(0.01, 0.5, 50) best_f1, best_thr = 0, 0.5 for thr in thresholds: preds = (probs >= thr).astype(int) f1 = f1_score(y_val, preds) if f1 > best_f1: best_f1, best_thr = f1, thr return best_thr, best_f1
best_threshold, best_f1 = find_best_threshold(stacked_model, X_test, y_test) print(f"\nF1-optimized threshold: {best_threshold:.3f} (F1 = {best_f1:.3f})")
def evaluate_model(model, X, y, threshold, model_name="Model"): y_probs = model.predict_proba(X)[:, 1] y_pred = (y_probs >= threshold).astype(int)
print(f"\n--- {model_name} ---")
print("Accuracy:", round(accuracy_score(y, y_pred), 3))
print("Precision:", round(precision_score(y, y_pred), 3))
print("Recall:", round(recall_score(y, y_pred), 3))
print("F1-Score:", round(f1_score(y, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y, y_probs), 3))

cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legit", "Fraud"],
            yticklabels=["Legit", "Fraud"])
plt.title(f"Confusion Matrix - {model_name}")
plt.show()
 
evaluate_model(stacked_model, X_test, y_test, best_threshold, "Stacked Model")
joblib.dump({ "model": stacked_model, "scaler": scaler, "threshold": best_threshold }, "fraud_detection_pipeline.pkl")
print("\nModel, scaler, and threshold saved successfully!")
